# Iron Correction Test: Does Subtracting Iron Absorption Recover Expected MAC?

**Hypothesis**: HIPS F_abs = MAC_BC * EC + MAC_Fe * Iron + residual

If iron oxide absorption on the filter explains the HIPS-FTIR discrepancy:
1. A multivariate regression of F_abs on EC and Iron should yield MAC_BC close to 10 m²/g
2. Subtracting the iron-proportional absorption from F_abs should recover slope ≈ 10 and intercept ≈ 0
3. The correction should work better in Dry season (most dust) than Kiremt (least dust)
4. Season-specific slopes should converge after correction

**Key question from slides**: Is the iron-dust mechanism sufficient to fully explain
both the ~28 Mm⁻¹ intercept AND the 0.40 slope compression, or are there secondary
organic aerosol effects yet to be isolated?

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import statsmodels.api as sm

FTIR_HIPS_DIR = Path('/Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem')
sys.path.insert(0, str(FTIR_HIPS_DIR / 'scripts'))
from config import MAC_VALUE
from data_matching import load_filter_data, pivot_filter_by_id

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'axes.spines.top': False, 'axes.spines.right': False,
})

SEASON_MAP = {10:'Dry',11:'Dry',12:'Dry',1:'Dry',2:'Dry',
              3:'Belg',4:'Belg',5:'Belg',6:'Kiremt',7:'Kiremt',8:'Kiremt',9:'Kiremt'}
SO = ['Dry','Belg','Kiremt']
SC = {'Dry':'#E67E22','Belg':'#27AE60','Kiremt':'#3498DB'}

OUT = 'output/plots/iron_correction'
os.makedirs(OUT, exist_ok=True)

print(f'Expected MAC_BC = {MAC_VALUE} m2/g')
print('Setup complete')

In [ ]:
# Load data with all fixes (date collapse, unit conversion)
filter_data = load_filter_data()

# Get HIPS, FTIR, and iron on the same rows via date-collapse
chem = pivot_filter_by_id(filter_data, 'ETAD', params=[
    'EC_ftir', 'HIPS_Fabs',
    'ChemSpec_Iron_PM2.5', 'ChemSpec_Silicon_PM2.5',
    'ChemSpec_Aluminum_PM2.5', 'ChemSpec_Calcium_PM2.5',
    'ChemSpec_Titanium_PM2.5',
])
chem['date'] = pd.to_datetime(chem['date'])
rename = {'chemspec_iron_pm2.5':'iron', 'chemspec_silicon_pm2.5':'silicon',
          'chemspec_aluminum_pm2.5':'aluminum', 'chemspec_calcium_pm2.5':'calcium',
          'chemspec_titanium_pm2.5':'titanium'}
chem.rename(columns=rename, inplace=True)

# Collapse by date so FTIR/HIPS (strip filters) merge with ChemSpec (teflon filters)
date_cols = [c for c in chem.columns if c != 'date']
chem = chem.groupby('date')[date_cols].first().reset_index()

# Iron is in ng/m3, convert to ug/m3 for consistency
chem['iron_ug'] = chem['iron'] / 1000

# Fine soil in ug/m3
chem['fine_soil'] = (2.20*chem['aluminum'] + 2.49*chem['silicon'] +
                     1.63*chem['calcium'] + 2.42*chem['iron'] +
                     1.94*chem['titanium']) / 1000

chem['hips_bc'] = chem['hips_fabs'] / MAC_VALUE
chem['Season'] = chem['date'].dt.month.map(SEASON_MAP)

# Working dataset: need all three (Fabs, EC, iron)
df = chem.dropna(subset=['hips_fabs', 'ftir_ec', 'iron']).copy()
print(f'Matched filters with Fabs + EC + Iron: {len(df)}')
for s in SO:
    print(f'  {s}: {len(df[df["Season"]==s])}')

print(f'\nUnits:')
print(f'  hips_fabs (Mm-1): {df["hips_fabs"].mean():.1f} +/- {df["hips_fabs"].std():.1f}')
print(f'  ftir_ec (ug/m3):  {df["ftir_ec"].mean():.2f} +/- {df["ftir_ec"].std():.2f}')
print(f'  iron (ng/m3):     {df["iron"].mean():.1f} +/- {df["iron"].std():.1f}')
print(f'  iron (ug/m3):     {df["iron_ug"].mean():.3f} +/- {df["iron_ug"].std():.3f}')

## Test 1: Baseline Regression (Uncorrected)

F_abs = slope * EC + intercept

The current problem: slope ≈ 4.0 (should be ~10) and intercept ≈ 28 Mm⁻¹ (should be ~0).

In [ ]:
# Baseline: Fabs vs EC (uncorrected)
slope_raw, int_raw, r_raw, p_raw, se_raw = stats.linregress(df['ftir_ec'], df['hips_fabs'])
print('=== UNCORRECTED: Fabs = slope * EC + intercept ===')
print(f'  slope = {slope_raw:.2f} m2/g  (expected: ~10)')
print(f'  intercept = {int_raw:.1f} Mm-1  (expected: ~0)')
print(f'  R2 = {r_raw**2:.3f}')
print(f'  n = {len(df)}')

# By season
print('\nBy season (uncorrected):')
for s in SO:
    sub = df[df['Season'] == s]
    if len(sub) >= 5:
        sl, it, r, p, se = stats.linregress(sub['ftir_ec'], sub['hips_fabs'])
        print(f'  {s}: slope={sl:.2f}, intercept={it:.1f}, R2={r**2:.3f}, n={len(sub)}')

## Test 2: Multivariate Regression

F_abs = MAC_BC * EC + MAC_Fe * Iron + constant

If iron explains the discrepancy, MAC_BC should be closer to 10 m²/g when iron is included.

In [ ]:
# Multivariate: Fabs = a * EC + b * Iron + c
# Use iron in ng/m3 (original units) for regression to get meaningful coefficient
X = sm.add_constant(df[['ftir_ec', 'iron']])
model = sm.OLS(df['hips_fabs'], X).fit()
print('=== MULTIVARIATE: Fabs = MAC_BC * EC + MAC_Fe * Iron + const ===')
print(model.summary2().tables[1].to_string())
print(f'\nInterpretation:')
print(f'  MAC_BC (EC coeff) = {model.params["ftir_ec"]:.2f} m2/g')
print(f'  MAC_Fe (Iron coeff) = {model.params["iron"]:.4f} Mm-1 per ng/m3 Iron')
print(f'  MAC_Fe (Iron coeff) = {model.params["iron"]*1000:.2f} Mm-1 per ug/m3 Iron')
print(f'  Constant = {model.params["const"]:.1f} Mm-1')
print(f'  R2 = {model.rsquared:.3f}  (was {r_raw**2:.3f} without iron)')
print(f'  R2_adj = {model.rsquared_adj:.3f}')

# Also try with fine_soil instead of iron alone
df_fs = df.dropna(subset=['fine_soil'])
if len(df_fs) > 10:
    X2 = sm.add_constant(df_fs[['ftir_ec', 'fine_soil']])
    model2 = sm.OLS(df_fs['hips_fabs'], X2).fit()
    print(f'\n--- Alternative: using Fine Soil instead of Iron ---')
    print(f'  MAC_BC = {model2.params["ftir_ec"]:.2f}, Fine Soil coeff = {model2.params["fine_soil"]:.3f}')
    print(f'  Constant = {model2.params["const"]:.1f}, R2 = {model2.rsquared:.3f}')

## Test 3: Subtract Iron Absorption → Check Corrected F_abs vs EC

F_abs_corrected = F_abs - MAC_Fe * Iron

Then regress F_abs_corrected vs EC. If the mechanism works, slope should approach 10 m²/g
and intercept should approach 0.

In [ ]:
# Subtract iron contribution using the coefficient from multivariate regression
mac_fe = model.params['iron']
df['fabs_corrected'] = df['hips_fabs'] - mac_fe * df['iron']

# Also try empirical sweep: what k gives slope closest to 10?
best_k, best_slope_diff = 0, 999
k_range = np.linspace(0, mac_fe * 3, 200)
sweep_results = []
for k in k_range:
    fabs_c = df['hips_fabs'] - k * df['iron']
    sl, it, r, p, se = stats.linregress(df['ftir_ec'], fabs_c)
    sweep_results.append({'k': k, 'slope': sl, 'intercept': it, 'R2': r**2})
    if abs(sl - 10) < best_slope_diff:
        best_slope_diff = abs(sl - 10)
        best_k = k
sweep_df = pd.DataFrame(sweep_results)

print(f'MAC_Fe from multivariate: {mac_fe:.5f} Mm-1 per ng/m3')
print(f'Best k for slope=10:      {best_k:.5f} Mm-1 per ng/m3')

# Corrected regression using multivariate MAC_Fe
sl_c, it_c, r_c, p_c, se_c = stats.linregress(df['ftir_ec'], df['fabs_corrected'])
print(f'\n=== CORRECTED (subtract MAC_Fe * Iron): Fabs_corr vs EC ===')
print(f'  slope = {sl_c:.2f} m2/g  (was {slope_raw:.2f})')
print(f'  intercept = {it_c:.1f} Mm-1  (was {int_raw:.1f})')
print(f'  R2 = {r_c**2:.3f}  (was {r_raw**2:.3f})')

# Corrected using best-k
df['fabs_bestk'] = df['hips_fabs'] - best_k * df['iron']
sl_bk, it_bk, r_bk, _, _ = stats.linregress(df['ftir_ec'], df['fabs_bestk'])
print(f'\n=== CORRECTED (best k for slope=10) ===')
print(f'  k = {best_k:.5f}')
print(f'  slope = {sl_bk:.2f}, intercept = {it_bk:.1f}, R2 = {r_bk**2:.3f}')

# Plot the sweep
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ax = axes[0]
ax.plot(sweep_df['k']*1000, sweep_df['slope'], 'b-', lw=2)
ax.axhline(10, color='red', ls='--', lw=1.5, label='Expected MAC = 10')
ax.axhline(slope_raw, color='gray', ls=':', lw=1, label=f'Uncorrected = {slope_raw:.1f}')
ax.axvline(mac_fe*1000, color='green', ls='--', alpha=0.7, label=f'Multivariate k')
ax.axvline(best_k*1000, color='orange', ls='--', alpha=0.7, label=f'Best k (slope=10)')
ax.set_xlabel('k (Mm-1 per ug/m3 Iron)')
ax.set_ylabel('Recovered slope (m2/g)')
ax.set_title('(a) Slope vs Iron correction factor k')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(sweep_df['k']*1000, sweep_df['intercept'], 'b-', lw=2)
ax.axhline(0, color='red', ls='--', lw=1.5, label='Target = 0')
ax.axhline(int_raw, color='gray', ls=':', lw=1, label=f'Uncorrected = {int_raw:.0f}')
ax.axvline(mac_fe*1000, color='green', ls='--', alpha=0.7)
ax.axvline(best_k*1000, color='orange', ls='--', alpha=0.7)
ax.set_xlabel('k (Mm-1 per ug/m3 Iron)')
ax.set_ylabel('Recovered intercept (Mm-1)')
ax.set_title('(b) Intercept vs correction factor k')
ax.legend(fontsize=8)

ax = axes[2]
ax.plot(sweep_df['k']*1000, sweep_df['R2'], 'b-', lw=2)
ax.axhline(r_raw**2, color='gray', ls=':', lw=1, label=f'Uncorrected R2 = {r_raw**2:.3f}')
ax.axvline(mac_fe*1000, color='green', ls='--', alpha=0.7)
ax.set_xlabel('k (Mm-1 per ug/m3 Iron)')
ax.set_ylabel('R2')
ax.set_title('(c) R2 vs correction factor k')
ax.legend(fontsize=8)

plt.suptitle('Iron Correction Sweep: Effect of subtracting k * Iron from Fabs', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/correction_sweep.png')
plt.show()

## Test 4: Before/After Scatter Plots — The Key Figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax_idx, (fabs_col, title, sl, it, r2) in enumerate([
    ('hips_fabs', f'(a) Uncorrected\nslope={slope_raw:.2f}, int={int_raw:.1f}, R$^2$={r_raw**2:.3f}',
     slope_raw, int_raw, r_raw**2),
    ('fabs_corrected', f'(b) Iron-corrected (multivariate k)\nslope={sl_c:.2f}, int={it_c:.1f}, R$^2$={r_c**2:.3f}',
     sl_c, it_c, r_c**2),
    ('fabs_bestk', f'(c) Iron-corrected (best k for MAC=10)\nslope={sl_bk:.2f}, int={it_bk:.1f}, R$^2$={r_bk**2:.3f}',
     sl_bk, it_bk, r_bk**2),
]):
    ax = axes[ax_idx]
    for s in SO:
        sub = df[df['Season'] == s]
        ax.scatter(sub['ftir_ec'], sub[fabs_col], s=40, alpha=0.55,
                  color=SC[s], edgecolors='k', linewidth=0.3, label=s, zorder=3)

    # Overall regression line
    xr = np.linspace(0, df['ftir_ec'].max() * 1.1, 50)
    ax.plot(xr, sl * xr + it, 'k--', lw=2, alpha=0.7, zorder=4)

    # Expected MAC=10 line (through origin)
    ax.plot(xr, 10 * xr, color='red', ls=':', lw=1.5, alpha=0.6, label='MAC=10 (expected)', zorder=2)

    # Per-season regression lines
    for s in SO:
        sub = df[df['Season'] == s]
        if len(sub) >= 5:
            sl_s, it_s, r_s, _, _ = stats.linregress(sub['ftir_ec'], sub[fabs_col])
            xr_s = np.linspace(sub['ftir_ec'].min(), sub['ftir_ec'].max(), 50)
            ax.plot(xr_s, sl_s * xr_s + it_s, color=SC[s], lw=2, alpha=0.8, zorder=4)

    ax.set_xlabel('FTIR EC (ug/m3)')
    ax.set_ylabel('F$_{abs}$ (Mm$^{-1}$)' if ax_idx == 0 else 'F$_{abs}$ corrected (Mm$^{-1}$)')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(0, None)
    ax.set_ylim(0, None)
    if ax_idx == 0:
        ax.legend(fontsize=8)

plt.suptitle('Iron Correction: Does subtracting iron absorption recover MAC = 10 m$^2$/g?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/before_after_scatter.png')
plt.show()

## Test 5: Season-Separated Regressions — Do Slopes Converge After Correction?

In [ ]:
# Compare season-separated regressions before and after correction
print(f'{"":12s} {"--- Uncorrected ---":>40s}  {"--- Iron-Corrected ---":>40s}')
print(f'{"Season":12s} {"slope":>8s} {"intercept":>10s} {"R2":>8s} {"n":>5s}'
      f'  {"slope":>8s} {"intercept":>10s} {"R2":>8s} {"n":>5s}')
print('-' * 100)

season_stats = []
for s in SO:
    sub = df[df['Season'] == s]
    if len(sub) >= 5:
        sl_u, it_u, r_u, _, _ = stats.linregress(sub['ftir_ec'], sub['hips_fabs'])
        sl_c2, it_c2, r_c2, _, _ = stats.linregress(sub['ftir_ec'], sub['fabs_corrected'])
        print(f'{s:12s} {sl_u:8.2f} {it_u:10.1f} {r_u**2:8.3f} {len(sub):5d}'
              f'  {sl_c2:8.2f} {it_c2:10.1f} {r_c2**2:8.3f} {len(sub):5d}')
        season_stats.append({
            'Season': s,
            'slope_raw': sl_u, 'int_raw': it_u, 'R2_raw': r_u**2,
            'slope_corr': sl_c2, 'int_corr': it_c2, 'R2_corr': r_c2**2,
            'n': len(sub),
        })

ss = pd.DataFrame(season_stats)
print(f'\nSlope convergence:')
print(f'  Uncorrected slope range: {ss["slope_raw"].min():.2f} – {ss["slope_raw"].max():.2f} '
      f'(spread = {ss["slope_raw"].max() - ss["slope_raw"].min():.2f})')
print(f'  Corrected slope range:   {ss["slope_corr"].min():.2f} – {ss["slope_corr"].max():.2f} '
      f'(spread = {ss["slope_corr"].max() - ss["slope_corr"].min():.2f})')
print(f'  Slope spread {"REDUCED" if (ss["slope_corr"].max()-ss["slope_corr"].min()) < (ss["slope_raw"].max()-ss["slope_raw"].min()) else "NOT reduced"} by correction')

print(f'\nIntercept reduction:')
print(f'  Uncorrected intercept range: {ss["int_raw"].min():.1f} – {ss["int_raw"].max():.1f}')
print(f'  Corrected intercept range:   {ss["int_corr"].min():.1f} – {ss["int_corr"].max():.1f}')

## Test 6: Residual Analysis — What Remains After Iron Correction?

If iron explains everything, residuals should be random. If there's a secondary effect
(organic aerosol, brown carbon), residuals may correlate with other variables.

In [ ]:
# Residuals from the multivariate model
df['residual'] = model.resid

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Residuals vs iron
ax = axes[0]
for s in SO:
    sub = df[df['Season'] == s]
    ax.scatter(sub['iron_ug'], sub['residual'], s=30, alpha=0.5, color=SC[s], label=s)
ax.axhline(0, color='k', ls='--', lw=1)
r_res_fe, p_res_fe = stats.pearsonr(df['iron_ug'], df['residual'])
ax.text(0.05, 0.95, f'r = {r_res_fe:.3f}\np = {p_res_fe:.2e}',
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel('Iron (ug/m3)')
ax.set_ylabel('Residual (Mm-1)')
ax.set_title('(a) Residual vs Iron\n(should be flat if iron fully explains)')
ax.legend(fontsize=8)

# (b) Residuals vs EC
ax = axes[1]
for s in SO:
    sub = df[df['Season'] == s]
    ax.scatter(sub['ftir_ec'], sub['residual'], s=30, alpha=0.5, color=SC[s], label=s)
ax.axhline(0, color='k', ls='--', lw=1)
r_res_ec, p_res_ec = stats.pearsonr(df['ftir_ec'], df['residual'])
ax.text(0.05, 0.95, f'r = {r_res_ec:.3f}\np = {p_res_ec:.2e}',
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel('FTIR EC (ug/m3)')
ax.set_ylabel('Residual (Mm-1)')
ax.set_title('(b) Residual vs EC\n(should be flat if MAC is correct)')
ax.legend(fontsize=8)

# (c) Residuals by season
ax = axes[2]
box_data = [df[df['Season'] == s]['residual'].dropna() for s in SO]
bp = ax.boxplot(box_data, tick_labels=SO, patch_artist=True, showfliers=True,
                flierprops=dict(marker='.', markersize=3, alpha=0.3))
for patch, s in zip(bp['boxes'], SO):
    patch.set_facecolor(SC[s])
    patch.set_alpha(0.7)
ax.axhline(0, color='k', ls='--', lw=1)
ax.set_ylabel('Residual (Mm-1)')
ax.set_title('(c) Residual by season\n(should be centered at 0, equal spread)')

# Kruskal-Wallis test for seasonal differences in residuals
groups = [df[df['Season'] == s]['residual'].dropna() for s in SO]
H, p_kw = stats.kruskal(*groups)
ax.text(0.05, 0.95, f'Kruskal-Wallis:\nH={H:.1f}, p={p_kw:.3e}',
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', fc='white', alpha=0.9))

plt.suptitle('Residual Analysis: What remains after EC + Iron correction?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/residual_analysis.png')
plt.show()

# Check if residuals correlate with fine_soil (broader dust metric)
df_fs = df.dropna(subset=['fine_soil'])
r_fs, p_fs = stats.pearsonr(df_fs['fine_soil'], df_fs['residual'])
print(f'\nResidual vs Fine Soil: r={r_fs:.3f}, p={p_fs:.3e}')
print(f'Residual vs Iron: r={r_res_fe:.3f}, p={p_res_fe:.3e}')
print(f'Residual vs EC: r={r_res_ec:.3f}, p={p_res_ec:.3e}')
print(f'\nResidual mean by season:')
for s in SO:
    sub = df[df['Season'] == s]['residual']
    print(f'  {s}: mean={sub.mean():.2f}, std={sub.std():.2f}')

# Is there a secondary effect?
resid_explained = 1 - (df['residual'].var() / df['hips_fabs'].var())
print(f'\nFraction of Fabs variance explained by EC + Iron: {resid_explained:.1%}')
print(f'Remaining unexplained variance: {1-resid_explained:.1%}')

## Test 7: Summary Verdict

Does iron fully explain the HIPS-FTIR discrepancy, or are secondary effects present?

In [ ]:
# Summary verdicts
print('=' * 80)
print('IRON CORRECTION TEST — SUMMARY')
print('=' * 80)

print(f'\n1. UNCORRECTED BASELINE:')
print(f'   Fabs = {slope_raw:.2f} * EC + {int_raw:.1f}')
print(f'   Slope is {slope_raw:.2f} m2/g (expected 10) → {slope_raw/10*100:.0f}% of expected')
print(f'   Intercept is {int_raw:.1f} Mm-1 (expected 0)')

print(f'\n2. MULTIVARIATE REGRESSION:')
print(f'   Fabs = {model.params["ftir_ec"]:.2f} * EC + {model.params["iron"]:.5f} * Iron + {model.params["const"]:.1f}')
print(f'   MAC_BC = {model.params["ftir_ec"]:.2f} m2/g ({"CLOSER" if abs(model.params["ftir_ec"]-10) < abs(slope_raw-10) else "NOT closer"} to 10)')
print(f'   R2 improved from {r_raw**2:.3f} to {model.rsquared:.3f}')

print(f'\n3. IRON CORRECTION EFFECT:')
print(f'   Slope: {slope_raw:.2f} → {sl_c:.2f} m2/g (change: {(sl_c-slope_raw)/slope_raw*100:+.0f}%)')
print(f'   Intercept: {int_raw:.1f} → {it_c:.1f} Mm-1 (change: {(it_c-int_raw)/int_raw*100:+.0f}%)')

print(f'\n4. SEASON CONVERGENCE:')
spread_raw = ss['slope_raw'].max() - ss['slope_raw'].min()
spread_corr = ss['slope_corr'].max() - ss['slope_corr'].min()
print(f'   Slope spread: {spread_raw:.2f} → {spread_corr:.2f} ({"CONVERGED" if spread_corr < spread_raw else "DID NOT CONVERGE"})')

print(f'\n5. VARIANCE EXPLAINED:')
print(f'   EC alone: {r_raw**2:.1%}')
print(f'   EC + Iron: {model.rsquared:.1%}')
print(f'   Remaining unexplained: {1-model.rsquared:.1%}')

print(f'\n6. VERDICT:')
if model.params['ftir_ec'] > 7 and model.rsquared > r_raw**2 + 0.02:
    print('   IRON CORRECTION WORKS: MAC_BC recovers closer to expected, R2 improves')
    if 1-model.rsquared > 0.10:
        print(f'   BUT {1-model.rsquared:.0%} variance remains → secondary effects likely present')
        print('   Candidates: brown carbon, other mineral absorbers, filter loading artifacts')
    else:
        print('   Iron + EC explain nearly all variance — mechanism is sufficient')
elif model.params['iron'] > 0 and model.pvalues['iron'] < 0.05:
    print('   IRON IS SIGNIFICANT but does not fully recover MAC = 10')
    print('   Secondary effects are present and substantial')
else:
    print('   IRON DOES NOT SIGNIFICANTLY IMPROVE the model')
    print('   The HIPS-FTIR discrepancy may have a different primary cause')